In [0]:
%pip install openai unidecode gspread==5.12.4 tiktoken mlflow
%restart_python
%load_ext autoreload
%autoreload 2 

In [0]:
from abc import ABC, abstractmethod
import time
import mlflow
from contextlib import contextmanager
import json
import pandas as pd
import datetime
import re
import openai
from openai import OpenAI
import gspread
import random
import logging
import os
from unidecode import unidecode
from pathlib import Path
import pyspark
from pyspark.sql.functions import *
from functools import reduce
from typing import *
import tiktoken
import sys


# Add the parent directory of 'localizers' to sys.path
project_root = Path('/Workspace/Users/krista@jamcity.com/CentralizedLocalizationWorkflow/localizers')
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

### Localizer Modules 
from general_config import *
from ml_tracker import *
from generic_localizer import *
from dmm_in_game_localizer import *
from in_game_config import *
from context_enrichment import *
#from dmm_humorous_guidelines import *


In [0]:
%run "./authenticationScript"

In [0]:
#TODO update authenticatinon script and secrets!!
gsheet_client = get_gspread_client_from_secret_old()
gpt_client = get_model_client() 

In [0]:
# Get request from widgets
request = get_request()


# GG cfg
cfg = {
    "input": {"required_tabs": ["input","output"], 
              "input_headers": ["KEY", 'Feature name / Content type', 'en_US', "Feature / Content", 'char_limit'], 
              "output_headers": ['token','en_US','char_limit', *DMM_LANG_CDS]
    },
    "char_limit_policy": "loose",
    "tracking_sheet_url": "",
}


# Generic Configuration
#cfg = {
#    "input": {"required_tabs": ["input","output"]},
#    "char_limit_policy": "loose",
#}

In [0]:
# Purpose is to take an input with some explicitly known columns ['en_US','char_limit'(optional),'context', 'token','other context']

In [0]:
localizer =DMMInGameLocalizer(request = request, 
                            gsheet_client =gsheet_client, 
                            gpt_client = gpt_client, 
                            cfg=cfg)

In [0]:
localizer.validate_inputs()

In [0]:
inputs = localizer.load_inputs()

In [0]:
prepped, prepped_item_desc = localizer.preprocess(inputs)

In [0]:
#
# 
# Now preproces
# 
# 
prepped_item_desc

In [0]:
def run_context_enrichment(prompt):

        MODEL = "gpt-5.4"
        temperature = 0.05

        response = gpt_client.chat.completions.create(
                model=MODEL, 
                messages=prompt,
                temperature=temperature  # adjust for creativity vs. stability
        )

        output = response.choices[0].message.content
        usage = response.usage
        
        return output, usage

def _parse_model_json_block(raw_output: str):
        """
        Clean and parse a JSON-like string from a model output wrapped in markdown code block.
        """
        try:
            # Strip markdown-style code block markers like ```json ... ```
            cleaned = re.sub(r"^```json|```$", "", raw_output.strip(), flags=re.IGNORECASE).strip()
            # Remove escaped newlines
            # PA 2025-11-06: TEMPORARY FIX, NEEDS MORE ANALYSIS - example: "<size=150%> %v0% \n<size=100%>off!". -->
            #cleaned = cleaned.replace("\\n", "").replace("\n", "").strip()
            # PA 2025-11-06: TEMPORARY FIX, NEEDS MORE ANALYSIS - example: "<size=150%> %v0% \n<size=100%>off!". <--
            loaded = json.loads(cleaned)
        except Exception as e:
            raise ValueError(f"Could not parse JSON: {e}")

        if isinstance(loaded, str):
            try:
                return json.loads(loaded)
            except Exception as e:
                raise ValueError(f"Could not parse inner JSON: {e}")
        return loaded
preprocess_prompt = preprocess_item_desc_DMM(localizer.item_desc)
output, usage = run_context_enrichment(preprocess_prompt)

In [0]:
def write_preprocessed(sh, parsed_df):
    num_cols= len(list(parsed_df.columns))
    num_rows = len(parsed_df.values.tolist())+1
    try:
        sh.add_worksheet('preprocessed', rows=num_rows, cols=num_cols)
    except Exception as e:
        pass
    preprocessed = sh.worksheet('preprocessed')

    headers = parsed_df.columns.tolist()
    out_data = parsed_df.values.tolist()


    number_to_letter = {
        "1": "A",  "2": "B",  "3": "C",  "4": "D",  "5": "E",
        "6": "F",  "7": "G",  "8": "H",  "9": "I", "10": "J",
        "11": "K", "12": "L", "13": "M", "14": "N", "15": "O",
        "16": "P", "17": "Q", "18": "R", "19": "S", "20": "T",
        "21": "U", "22": "V", "23": "W", "24": "X", "25": "Y",
        "26": "Z", "27": "AA", "28": "AB", "29": "AC", "30": "AD",
        "31": "AE", "32": "AF", "33": "AG", "34": "AH", "35": "AI",
        "36": "AJ", "37": "AK", "38": "AL", "39": "AM", "40": "AN",
        "41": "AO", "42": "AP", "43": "AQ", "44": "AR", "45": "AS",
        "46": "AT", "47": "AU", "48": "AV", "49": "AW", "50": "AX",
        "51": "AY", "52": "AZ", "53": "BA", "54": "BB", "55": "BC",
        "56": "BD", "57": "BE", "58": "BF", "59": "BG", "60": "BH",
    }
    letter_range = number_to_letter[str(len(headers))]
    data_range = f"A2:{letter_range}{len(out_data)+1}"


    preprocessed.batch_update([{
                    'range':f"A1:{letter_range}1", 'values':[headers]
                    },
                    {
                    'range':data_range, 'values':out_data
                    }])
    
    return "Done writing preprocessed!"

In [0]:
localizer.__dict__

In [0]:
preprocess_parsed = pd.DataFrame(_parse_model_json_block(output))
write_preprocessed(localizer.sh, preprocess_parsed)

In [0]:
def _generate_payload_humourous(preprocessed):
        payload = []
        for _, r in preprocessed.iterrows():
            #en = r.get("en_US", "") or ""
            key = r.get("key")
            item = {
                'key':key,
                'object': r.get("object",""),
                'descriptor': r.get("descriptor",""),
                'additional_context': r.get("additional_context",""),
                'simple_description_en': r.get("simple_description_en",""),
                # keep payload tiny; include only relevant hints
            }

            payload.append(item)


        prepped_payload = json.dumps(payload)
        return prepped_payload
    


def _generate_humorous_prompt_helper(language:str, 
                                #game:str, 
                                prepped_payload:str, 
                                humor_mode:str,
                                )->List[Dict[str, Any]]:
        #preprocessed_enriched = self._preprocess_for_humorous(prepped)

        if humor_mode == "object_quip":
            base = QUIP_PROMPT
            base += f"TARGET Language = {language}"
            base += QUIP_LANGUAGE_GUIDE[language] #TODO, check how this is keeyed
            base += OUTPUT_FORMAT_QUIP
        elif humor_mode == "fun_translation":
            base = FUN_DESCRIPTION_PROMPT
            base += f"TARGET Language = {language}"
            base += FUN_DESCRIPTIONS_LANGUAGE_GUIDE[language]
            base += OUTPUT_FORMAT_FUN_DESCRIPTION
        else:
            raise Exception(f"Mode {humor_mode} not supported")

        ### prepped needs to be preprocesed_enriched
        #preprocessed_payload = self._generate_payload_humourous(preprocessed_enriched)
        return [
            {"role": "system", "content": base},
            {"role": "user",   "content": prepped_payload}
        ]


prepped_payload = _generate_payload_humourous(preprocess_parsed)



prompts_objects = []
for language in localizer.languages:
    prompt = _generate_humorous_prompt_helper(language, 
                                prepped_payload, 
                                "object_quip")
    prompts_objects.append(prompt)

groups_objects = dict(zip(localizer.languages, prompts_objects))



prompts_descriptions = []
for language in localizer.languages:
    prompt = _generate_humorous_prompt_helper(language, 
                                prepped_payload, 
                                "fun_translation")
    prompts_descriptions.append(prompt)

groups_descriptions = dict(zip(localizer.languages, prompts_descriptions))

In [0]:
groups_objects

In [0]:
def run_context_enrichment(prompt):

        MODEL = "gpt-5.4"
        temperature = 0.05

        response = gpt_client.chat.completions.create(
                model=MODEL, 
                messages=prompt,
                temperature=temperature  # adjust for creativity vs. stability
        )

        output = response.choices[0].message.content
        usage = response.usage
        
        return output, usage
    
def _call_model_batch(prompt:str):
        """
        Must return (outputs, usage_dict) where usage_dict includes:
          {'prompt_tokens': int, 'completion_tokens': int}
        """
        MODEL = "gpt-4o"
        temperature = 0.05

        response = gpt_client.chat.completions.create(
                model=MODEL, 
                messages=prompt,
                temperature=0.05  # adjust for creativity vs. stability
        )

        output = response.choices[0].message.content
        usage = response.usage
        return output, usage


In [0]:
output_dfs_objects = []
for group, prompt in groups_objects.items():
    output, usage = _call_model_batch(prompt)
    parsed = _parse_model_json_block(output)
    output_dfs_objects.append(pd.DataFrame(parsed))

In [0]:
output_dfs_descriptions = []
for group, prompt in groups_descriptions.items():
    output, usage = _call_model_batch(prompt)
    parsed = _parse_model_json_block(output)
    output_dfs_descriptions.append(pd.DataFrame(parsed))

In [0]:
groups_objects.keys()

In [0]:
output_dfs_descriptions[6].display()

In [0]:
output_dfs_objects[6].display()

In [0]:
list(groups_objects.keys())

In [0]:
prompts = localizer.build_prompts(prepped)
#prompts_item_desc_quip = localizer.build_prompts_humorous(prepped_item_desc, humor_mode = "object_quip")
#prompts_item_desc_fun_translation = localizer.build_prompts_humorous(prepped_item_desc, humor_mode = "fun_translation")

In [0]:
prompts_item_desc_quip = localizer.build_prompts_humorous(prepped_item_desc, humor_mode = "object_quip")

In [0]:
base = QUIP_PROMPT
base += QUIP_LANGUAGE_GUIDE["French"] #TODO, check how this is keeyed
base += OUTPUT_FORMAT_QUIP

In [0]:
localizer.preprocessed_enriched

In [0]:
base

In [0]:
prompts_item_desc_quip = localizer.build_prompts_humorous(prepped_item_desc, humor_mode = "object_quip")

In [0]:
prompts_item_desc_fun_translation = localizer.build_prompts_humorous(prepped_item_desc, humor_mode = "fun_translation")

In [0]:
prompts_item_desc_fun_translation['French']

In [0]:
prompts_item_desc_quip.keys()

In [0]:
groups = localizer.build_prompts(prepped)

In [0]:
raw_results = localizer.translate(prompts)


In [0]:
raw_results_quip = localizer.translate(prompts_item_desc_quip)

In [0]:

#raw_results_quip = localizer.translate(prompts_item_desc_quip)
raw_results_fun_translation = localizer.translate(prompts_item_desc_fun_translation)

In [0]:
processed = localizer.postprocess(raw_results)

In [0]:
processed_quips = localizer.postprocess(raw_results_quip)
processed_fun_translation = localizer.postprocess(raw_results_fun_translation)

In [0]:
#localizer._merge_outputs_by_language(processed)

In [0]:
localizer.write_outputs(processed)

In [0]:
merged_quips = localizer._merge_outputs_by_language(processed_quips)
merged_descriptions = localizer._merge_outputs_by_language(processed_fun_translation)
#

In [0]:
try:
    localizer.sh.add_worksheet("output_quips", rows=100, cols=26)
except:
    pass
quips_wksht = localizer.sh.worksheet('output_quips')
number_to_letter = {
            "1": "A",  "2": "B",  "3": "C",  "4": "D",  "5": "E",
            "6": "F",  "7": "G",  "8": "H",  "9": "I", "10": "J",
            "11": "K", "12": "L", "13": "M", "14": "N", "15": "O",
            "16": "P", "17": "Q", "18": "R", "19": "S", "20": "T",
            "21": "U", "22": "V", "23": "W", "24": "X", "25": "Y",
            "26": "Z", "27": "AA", "28": "AB", "29": "AC", "30": "AD",
            "31": "AE", "32": "AF", "33": "AG", "34": "AH", "35": "AI",
            "36": "AJ", "37": "AK", "38": "AL", "39": "AM", "40": "AN",
            "41": "AO", "42": "AP", "43": "AQ", "44": "AR", "45": "AS",
            "46": "AT", "47": "AU", "48": "AV", "49": "AW", "50": "AX",
            "51": "AY", "52": "AZ", "53": "BA", "54": "BB", "55": "BC",
            "56": "BD", "57": "BE", "58": "BF", "59": "BG", "60": "BH",
        }
headers = merged_quips.columns.tolist()
out_data = merged_quips.values.tolist()

letter_range = number_to_letter[str(len(headers))]

headers_range = f"A1:{letter_range}1"
data_range = f"A2:{letter_range}{len(out_data)+1}"
quips_wksht.clear()

quips_wksht.batch_update([{'range':headers_range, 'values':[headers]},
                    {'range':data_range, 'values':out_data}])



In [0]:
try:
    localizer.sh.add_worksheet("output_fun_translations", rows=100, cols=26)
except:
    pass
fun_translations_wksht = localizer.sh.worksheet('output_fun_translations')
number_to_letter = {
            "1": "A",  "2": "B",  "3": "C",  "4": "D",  "5": "E",
            "6": "F",  "7": "G",  "8": "H",  "9": "I", "10": "J",
            "11": "K", "12": "L", "13": "M", "14": "N", "15": "O",
            "16": "P", "17": "Q", "18": "R", "19": "S", "20": "T",
            "21": "U", "22": "V", "23": "W", "24": "X", "25": "Y",
            "26": "Z", "27": "AA", "28": "AB", "29": "AC", "30": "AD",
            "31": "AE", "32": "AF", "33": "AG", "34": "AH", "35": "AI",
            "36": "AJ", "37": "AK", "38": "AL", "39": "AM", "40": "AN",
            "41": "AO", "42": "AP", "43": "AQ", "44": "AR", "45": "AS",
            "46": "AT", "47": "AU", "48": "AV", "49": "AW", "50": "AX",
            "51": "AY", "52": "AZ", "53": "BA", "54": "BB", "55": "BC",
            "56": "BD", "57": "BE", "58": "BF", "59": "BG", "60": "BH",
        }
headers = merged_descriptions.columns.tolist()
out_data = merged_descriptions.values.tolist()

letter_range = number_to_letter[str(len(headers))]

headers_range = f"A1:{letter_range}1"
data_range = f"A2:{letter_range}{len(out_data)+1}"
fun_translations_wksht.clear()

fun_translations_wksht.batch_update([{'range':headers_range, 'values':[headers]},
                    {'range':data_range, 'values':out_data}])



In [0]:
defv write_outputs(merged, tab_name='output')->str: 

        #results = self._merge_outputs_by_language(post)
        #self.results = results
       #try:
            self.sh.a
       # except:
        #wksht = self.sh.worksheet(tab_name)

        number_to_letter = {
            "1": "A",  "2": "B",  "3": "C",  "4": "D",  "5": "E",
            "6": "F",  "7": "G",  "8": "H",  "9": "I", "10": "J",
            "11": "K", "12": "L", "13": "M", "14": "N", "15": "O",
            "16": "P", "17": "Q", "18": "R", "19": "S", "20": "T",
            "21": "U", "22": "V", "23": "W", "24": "X", "25": "Y",
            "26": "Z", "27": "AA", "28": "AB", "29": "AC", "30": "AD",
            "31": "AE", "32": "AF", "33": "AG", "34": "AH", "35": "AI",
            "36": "AJ", "37": "AK", "38": "AL", "39": "AM", "40": "AN",
            "41": "AO", "42": "AP", "43": "AQ", "44": "AR", "45": "AS",
            "46": "AT", "47": "AU", "48": "AV", "49": "AW", "50": "AX",
            "51": "AY", "52": "AZ", "53": "BA", "54": "BB", "55": "BC",
            "56": "BD", "57": "BE", "58": "BF", "59": "BG", "60": "BH",
        }
        headers = results.columns.tolist()
        out_data = results.values.tolist()

        letter_range = number_to_letter[str(len(headers))]
        
        headers_range = f"A1:{letter_range}1"
        data_range = f"A2:{letter_range}{len(out_data)+1}"
        wksht.clear()

        wksht.batch_update([{'range':headers_range, 'values':[headers]},
                            {'range':data_range, 'values':out_data}])

        return "Done!"



In [0]:
pd.DataFrame(processed_fun_translation)

In [0]:
localizer.write_outputs(processed_quips, tab_name = "quips")

In [0]:
mlflow.end_run()

In [0]:
results = localizer.run()

In [0]:
dbutils.notebook.exit(json.dumps(results))